# 01 — Data Cleaning

**Input**: `data/raw_jobs.csv` (from crawler)
**Output**: `data/cleaned_jobs.csv`

Pipeline:
1. Load raw data
2. Check missing values
3. Remove duplicates
4. Parse locations into city/district
5. Split salary into min/max; flag salary disclosure
6. Normalize work model
7. Normalize experience level
8. Standardize skills using dictionary
9. Save cleaned output

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1. Load Raw Data

In [ ]:
DATA = Path('data')
df = pd.read_csv(DATA / 'raw_jobs.csv')
print(f'Loaded {len(df)} rows')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
miss_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
miss_df[miss_df['missing'] > 0].sort_values('missing', ascending=False)

## 3. Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['job_url'])
print(f'Removed {before - len(df)} duplicates ({before} -> {len(df)})')

## 4. Parse Location into City

In [ ]:
CITY_MAP = {
    'HCM': 'Ho Chi Minh City',
    'HN': 'Ha Noi',
    'Đà Nẵng': 'Da Nang',
    'Hải Phòng': 'Hai Phong',
    'Cần Thơ': 'Can Tho',
    'Huế': 'Hue',
    'Nha Trang': 'Nha Trang',
    'Online': 'Remote',
    'Toàn Quốc': 'Nationwide',
}

def normalize_city(raw):
    if pd.isna(raw) or raw == '':
        return ''
    first = raw.split('/')[0].split('-')[0].split(',')[0].strip()
    return CITY_MAP.get(first, first)

df['city'] = df['location'].apply(normalize_city)
print('Top cities:')
df['city'].value_counts().head(10)

## 5. Flag Salary Disclosure

In [ ]:
df['is_salary_public'] = df['salary_min'].notna()
pct_disclosed = df['is_salary_public'].mean() * 100
print(f'Salary disclosed: {pct_disclosed:.1f}% of jobs')
print(f'Before/after sample:')
df[['job_title', 'salary_min', 'salary_max', 'is_salary_public']].head(5)

## 6. Normalize Work Model

In [ ]:
WM_MAP = {'Remote': 'Remote', 'Hybrid': 'Hybrid', 'Onsite': 'Onsite'}
df['work_model'] = df['work_model'].map(WM_MAP).fillna('Unknown')
print(df['work_model'].value_counts())

## 7. Normalize Experience Level

In [ ]:
LEVELS = ['Internship', 'Fresher', 'Junior', 'Mid-level', 'Senior', 'Manager']
df['experience_level'] = pd.Categorical(
    df['experience_level'].fillna('Unknown'),
    categories=['Unknown'] + LEVELS,
    ordered=True
)
print(df['experience_level'].value_counts().sort_index())

## 8. Standardize Skills

In [ ]:
# Load skill dictionary
skill_dict = pd.read_csv(DATA / 'skills_dictionary.csv')
skill_map = dict(zip(skill_dict['raw_skill'].str.lower(), skill_dict['normalized_skill']))
cat_map = dict(zip(skill_dict['normalized_skill'], skill_dict['category']))

def normalize_skills(skills_str):
    if pd.isna(skills_str) or skills_str.strip() == '':
        return []
    skills = [s.strip().lower() for s in re.split(r'[;,]', skills_str) if s.strip()]
    normalized = list(dict.fromkeys(skill_map.get(s, s) for s in skills))
    return normalized

df['skills_list'] = df['skills'].apply(normalize_skills)
df['skill_count'] = df['skills_list'].apply(len)

# Show skill cleaning examples
print('Skill normalization examples:')
for raw_skill in ['python', 'js', 'reactjs', 'node.js', 'lập trình python']:
    norm = skill_map.get(raw_skill, raw_skill)
    cat = cat_map.get(norm, 'Other')
    print(f'  {raw_skill:20s} -> {norm:20s} [{cat}]')

## 9. Save Cleaned Data

In [ ]:
clean = df[[
    'job_title', 'company_name', 'city', 'salary_min', 'salary_max',
    'is_salary_public', 'experience_level', 'work_model', 'skills_list',
    'skill_count', 'posted_date', 'job_url', 'description'
]].copy()

clean.insert(0, 'job_id', range(1, len(clean) + 1))

# Convert skills_list to string for CSV
clean['skills'] = clean['skills_list'].apply(lambda x: '; '.join(x))
clean = clean.drop(columns=['skills_list'])

clean.to_csv(DATA / 'cleaned_jobs.csv', index=False, encoding='utf-8-sig')
print(f'Saved {len(clean)} rows to {DATA / "cleaned_jobs.csv"}')
print(f'\nCleaned columns: {list(clean.columns)}')
clean.head(3)